# Fine-tuning the UI Motion & Design Showcase Retriever

End-to-end, reproducible pipeline that fine-tunes the sentence encoder behind
`search_design_showcases.py`, measures retrieval quality **before vs after** on a
held-out benchmark, and exports a lightweight ONNX encoder + vector index.

**What actually ships.** The deployed index (`design_showcases/design_embeddings.npy`)
is **384-dimensional**, produced by a fine-tuned
[`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
(22M params, Apache-2.0). It runs on CPU and exports to a ~23MB int8 ONNX encoder.
Google's **EmbeddingGemma-2** (768-d, ~740M params, multimodal) is an *optional upgrade
path* for GPU machines with `transformers>=5.18` / `sentence-transformers>=6.1`; it is
gated and far heavier, so the default and shipped model is MiniLM. This notebook trains
the model that matches the shipped index.

**The recipe.** Multiple-Negatives Ranking Loss with *query-style* anchors derived from
each showcase description (always semantically rich), aligned to the index text
`"{title}: {description}"`. Brand-only titles (<=2 content words) are never used as
anchors, which avoids the false-negative noise that a naive title-anchor recipe injects.

In [ ]:
# [1] Dependencies (CPU works out of the box; GPU is used automatically if present)
!pip install -q -U sentence-transformers transformers datasets accelerate scikit-learn
!pip install -q onnx onnxruntime tokenizers numpy

In [ ]:
# [2] Locate the repo and load the unified dataset (no Google Drive required)
import os, sys, json, re, random
import numpy as np

# Run from the repo root, or set REPO to a clone of steoward/unfuckin-slop.
REPO = os.environ.get('UNFSLOP_REPO', os.getcwd())
if not os.path.exists(os.path.join(REPO, 'design_showcases', 'showcases_meta.json')):
    !git clone https://github.com/steoward/unfuckin-slop /content/unfuckin-slop
    REPO = '/content/unfuckin-slop'
sys.path.insert(0, os.path.join(REPO, 'eval'))
sys.path.insert(0, REPO)

META_PATH = os.path.join(REPO, 'design_showcases', 'showcases_meta.json')
with open(META_PATH, encoding='utf-8') as f:
    meta = json.load(f)
print(f'Loaded {len(meta)} showcases from {META_PATH}')
print('  3D/spatial :', sum(1 for m in meta if m.get('is_3d')))
print('  liquid glass:', sum(1 for m in meta if m.get('is_liquid_glass')))
print('  with video  :', sum(1 for m in meta if m.get('video_url')))

def index_text(m):
    return f"{m['title']}: {m['description']}"

In [ ]:
# [3] Build query-style contrastive pairs (the improved recipe)
TRAIL = {'a','an','the','and','or','for','with','of','in','on','to','is','into','over',
         'from','by','at','as','that','this','its','featuring','showcasing','showing',
         'displaying','demonstrating','followed','while','during','across','around','between'}

def desc_anchor(desc, max_words=14):
    d = re.sub(r'(Made by [^.]+\.|Curated from the internet\.)', '', desc).strip()
    d = re.sub(r'^(A|An|The)\s+', '', d)
    d = re.sub(r'^(clean|sleek|minimalist|modern|interactive|animated|dynamic|dark[- ]mode|light[- ]mode)[, ]+', '', d, flags=re.I)
    words = d.split()[:max_words]
    while words and words[-1].lower().strip('.,;:') in TRAIL:
        words.pop()
    return ' '.join(words).lower().rstrip('.,;:')

def clean_title(title):
    return re.sub(r'\s+', ' ', re.sub(r'[*_`#>]', '', title)).strip()

def title_anchor(title):
    t = clean_title(title)
    words = [w for w in re.split(r'[\s:,.]+', t) if w]
    return t if len(words) >= 3 else None

# Three curricula phases, each one epoch over all items.
phases = [
    ('A_desc14', [(desc_anchor(m['description'], 14), index_text(m)) for m in meta]),
    ('B_desc20', [(desc_anchor(m['description'], 20), index_text(m)) for m in meta]),
    ('C_title_or_desc', [((title_anchor(m['title']) or desc_anchor(m['description'], 14)), index_text(m)) for m in meta]),
]
print('Example anchor -> positive:')
print(' ', phases[0][1][0][0])
print(' ', phases[0][1][0][1][:120])

In [ ]:
# [4] Fine-tune all-MiniLM-L6-v2 with MultipleNegativesRankingLoss
# Train 3 curriculum phases, score each on the dev split, and keep the best.
import torch
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from datasets import Dataset
import evaluate as ev

BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
OUT_ROOT = os.environ.get('UNFSLOP_FT_OUT', os.path.join(REPO, 'ft_models'))
os.makedirs(OUT_ROOT, exist_ok=True)
use_cpu = not torch.cuda.is_available()
print('Device:', 'CPU' if use_cpu else torch.cuda.get_device_name(0))

bench = ev.load_bench()
texts = ev.doc_texts(meta)

def dev_score(m):
    docs = m.encode(texts, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)
    sc = ev.system_semantic(meta, bench, m, docs)
    return ev.aggregate(sc, bench, 'dev')

model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = 128
loss = MultipleNegativesRankingLoss(model)

log = []
for i, (name, pairs) in enumerate(phases, 1):
    ds = Dataset.from_dict({'anchor': [p[0] for p in pairs], 'positive': [p[1] for p in pairs]})
    out_dir = os.path.join(OUT_ROOT, f'queryv2_phase{i}')
    targs = SentenceTransformerTrainingArguments(
        output_dir=out_dir, num_train_epochs=1, per_device_train_batch_size=32,
        learning_rate=1e-5, warmup_steps=0.1, fp16=False, use_cpu=use_cpu,
        logging_steps=10, save_strategy='no', report_to='none')
    trainer = SentenceTransformerTrainer(model=model, train_dataset=ds, loss=loss, args=targs)
    print(f'--- phase {i}/3 ({name}) ---')
    trainer.train()
    model.save_pretrained(out_dir)
    d = dev_score(model)
    log.append({'dir': out_dir, 'dev': d})
    print(f'dev after {name}: {d}')

BEST = max(log, key=lambda r: (r['dev']['MRR@10'], r['dev']['nDCG@5']))['dir']
print('Best dev checkpoint:', BEST)

In [ ]:
# [5] BEFORE vs AFTER on the held-out benchmark (eval/benchmark.json)
# 'base' = un-tuned MiniLM; BEST = the fine-tuned checkpoint. dev selects, test reports.
import evaluate as ev
queries = ev.load_bench()
texts = ev.doc_texts(meta)

base_model = SentenceTransformer(BASE_MODEL)
base_docs = ev.encode_cached(base_model, texts, 'base_minilm_docs')
ft_model = SentenceTransformer(BEST)
ft_docs = ev.encode_cached(ft_model, texts, 'notebook_ft_docs')

base_scores = ev.system_semantic(meta, queries, base_model, base_docs)
ft_scores = ev.system_semantic(meta, queries, ft_model, ft_docs)

for label, sc in [('BEFORE (base MiniLM)', base_scores), ('AFTER (fine-tuned)', ft_scores)]:
    print(f'\n{label}')
    for split in ('dev', 'test'):
        print(f'  {split:5s}', ev.aggregate(sc, queries, split))

In [ ]:
# [6] Export the int8 ONNX encoder and rebuild the vector index
!python {os.path.join(REPO, 'eval', 'export_encoder.py')} --checkpoint {BEST}
!python {os.path.join(REPO, 'semantic_index.py')} rebuild
emb = np.load(os.path.join(REPO, 'design_showcases', 'design_embeddings.npy'))
print('Index shape:', emb.shape, '| unit norms:', bool(np.allclose(np.linalg.norm(emb, axis=1), 1.0, atol=1e-2)))

In [ ]:
# [7] Query the deployed hybrid search (lexical + semantic)
import search_design_showcases as sds
for q in ['liquid glass refraction with chromatic aberration over bold typography',
          '3d interactive spatial hero with generative particle orbit in three.js',
          'dark analytics dashboard with revenue line chart and KPI metric cards',
          'minimalist dark mode date and time picker']:
    print(f'\nQ: {q}')
    for r in sds.search(q, limit=3):
        print(f"  [{r['relevance_score']:.3f}] {r['title']}  ({r.get('category','UI')})")